# 02 — Build chip datasets

Chips every city into 256×256 patches, writes memory-mapped uint16 arrays per split, computes fixed train-split normalization stats, and pushes one zip per split plus `manifest.json` to Drive. Run **once** — the outputs are permanent on Drive.

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git"
REPO = "/content/s2-change-detection"
BRANCH = "chennai"          # the whole 00-07 series runs this branch
if os.path.exists(REPO):
    !git -C {REPO} checkout {BRANCH}
    !git -C {REPO} pull --ff-only origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!pip install -q .
import sys
sys.path.insert(0, os.path.join(REPO, "src"))
# drop cached s2cd modules so a just-pulled fix takes effect in THIS kernel
for k in [k for k in list(sys.modules) if k == "s2cd" or k.startswith("s2cd.")]:
    del sys.modules[k]


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
from s2cd.data.build import build

oscd_root = store.extract_oscd()
manifest = build(oscd_root, "configs/data.yaml", drive=store,
                 scratch=store.scratch)


In [ ]:
import json
print(json.dumps({k: v for k, v in manifest.items() if k != "stats"}, indent=1))
print("mean:", [round(m, 4) for m in manifest["stats"]["mean"]])
print("std :", [round(s, 4) for s in manifest["stats"]["std"]])


In [ ]:
# One training chip, visualized: earlier image / later image / change mask.
# This is the exact sample the network sees (256x256, 10 bands; RGB shown).
import numpy as np
import matplotlib.pyplot as plt

chips = store.scratch / "chips" / "train"
t0 = np.load(chips / "t0.npy", mmap_mode="r")   # (N, 10, 256, 256) uint16
t1 = np.load(chips / "t1.npy", mmap_mode="r")
mask = np.load(chips / "mask.npy", mmap_mode="r")

# class balance over the whole train split - why Dice is in the loss
print(f"train chips: {len(mask)} | change pixels: {float(mask.mean()):.1%} "
      "(rare class -> Dice term in the loss)")

# the chip with the most change pixels, so the mask is actually visible
i = int(np.argmax(mask.sum(axis=(1, 2))))
bands_cfg = load_config("configs/data.yaml")["bands"]
rgb = [bands_cfg.index("B04"), bands_cfg.index("B03"), bands_cfg.index("B02")]
# a[rgb] (NOT a[[rgb]]): rgb is a list, so double brackets nest it as a
# 2-D index array -> (1, 3, H, W) and the transpose below fails.
stretch = lambda a: np.clip(a[rgb].astype(np.float32) / 3000.0, 0, 1).transpose(1, 2, 0)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(stretch(t0[i])); axes[0].set_title("Earlier image")
axes[1].imshow(stretch(t1[i])); axes[1].set_title("Later image")
axes[2].imshow(mask[i], cmap="gray"); axes[2].set_title("Ground-truth change mask")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()
